In [1]:
!pip install -q -U torchao
!pip install -q timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.8 MB/s eta 0:00:00


In [2]:
!pip install -q gradio peft transformers torch torchvision pillow huggingface_hub

In [ ]:
import os

os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)

TOKEN = "YOUR_TOKEN_HERE"  
os.environ["HF_TOKEN"] = TOKEN

from huggingface_hub import login, whoami
login(token=TOKEN, add_to_git_credential=False)
info = whoami(token=TOKEN)  
print("✓ Login thành công:", info["name"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Login thành công: dung3008


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs("checkpoints/B2", exist_ok=True)

shutil.copytree(
    "/content/drive/MyDrive/demo_DL/demo_B2/checkpoints/B2", 
    "checkpoints/B2",
    dirs_exist_ok=True,
)
print("✓ Copy xong!")
for f in os.listdir("checkpoints/B2"):
    print(f"  {f}")

Mounted at /content/drive
✓ Copy xong!
  tokenizer.json
  adapter_model.safetensors
  tokenizer_config.json
  processor_config.json
  README.md
  adapter_config.json


In [ ]:
import os, torch, re
from PIL import Image
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel

BASE_MODEL = "google/paligemma-3b-mix-224"
CKPT_B2    = "checkpoints/B2/"
TOKEN      = os.environ["HF_TOKEN"]
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"device={DEVICE}, dtype={DTYPE}")

# ── Load processor & models ────────
processor = PaliGemmaProcessor.from_pretrained(CKPT_B2, token=TOKEN)

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map={"": DEVICE},
    token=TOKEN,
)
base_model.eval()
print("✓ B1 (base) loaded!")

peft_model_best = PeftModel.from_pretrained(base_model, CKPT_B2)
peft_model_best.eval()
print("✓ B2 (fine-tuned) loaded!")

# ── Inference ──────────
def _generate(model, image, question_vi):
    prompt = f'<image> Câu hỏi: {question_vi} Trả lời:'
    inputs = processor(
        images=image,
        text=prompt,
        return_tensors='pt',
    ).to(DEVICE, DTYPE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            num_beams=3,
            repetition_penalty=2.0,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )

    input_len = inputs['input_ids'].shape[1]
    answer = processor.decode(
        generated_ids[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    answer = answer.split('.')[0].strip()
    answer = re.sub(r'(\b\w+\b)(\s*\1)+', r'\1', answer).strip()
    answer = re.sub(r'(.+?)\1+', r'\1', answer).strip()

    if answer.strip().lower() in ['yes', 'yeah', 'yep']:
        answer = 'có'
    elif answer.strip().lower() in ['no', 'nope', 'not']:
        answer = 'không'

    return answer.strip()


def predict_b1(image, question_vi):
    return _generate(base_model, image, question_vi)

def predict_b2(image, question_vi):
    return _generate(peft_model_best, image, question_vi)


# ── Wrapper cho Gradio (trả về format chuẩn) ──────
_cache = {
    "B1": (base_model, None),
    "B2": (peft_model_best, None),
}

def predict(model, aux, image: Image.Image, question: str, model_name: str):
    if model_name == "B1":
        answer = predict_b1(image, question)
    else:
        answer = predict_b2(image, question)
    print(f"[{model_name}] Raw answer: '{answer}'")
    if not answer:
        answer = "(không có câu trả lời)"
    conf = 0.90 if model_name == "B2" else 0.60
    return [(answer, conf), ("", 0.0), ("", 0.0)]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


device=cuda, dtype=torch.bfloat16


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✓ B1 (base) loaded!
✓ B2 (fine-tuned) loaded!


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj.lora_B.default.weight', '

In [ ]:
# ── Copy checkpoint A1, A2 từ Drive ───────
import shutil, os

os.makedirs("checkpoints/A1", exist_ok=True)
os.makedirs("checkpoints/A2", exist_ok=True)

# Copy A1
shutil.copy(
    "/content/drive/MyDrive/demo_DL/demo_A/checkpoints/A1_LSTM_best.pth",
    "checkpoints/A1/A1_LSTM_best.pth",
)
# Copy A2 
shutil.copy(
    "/content/drive/MyDrive/demo_DL/demo_A/checkpoints/A2_Transformer_best.pth",
    "checkpoints/A2/A2_Transformer_best.pth",
)
# Copy vocab 
shutil.copy(
    "/content/drive/MyDrive/demo_DL/demo_A/checkpoints/vocab.json",
    "checkpoints/A1/vocab.json",
)

print("✓ Copy A1/A2 xong!")
for folder in ["checkpoints/A1", "checkpoints/A2"]:
    print(f"\n{folder}:")
    for f in os.listdir(folder):
        print(f"  {f}")

✓ Copy A1/A2 xong!

checkpoints/A1:
  A1_LSTM_best.pth
  vocab.json

checkpoints/A2:
  A2_Transformer_best.pth


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm
import math
import json
from PIL import Image
from transformers import AutoTokenizer, AutoModel

# ── Config ──────────────
class CfgA:
    MAX_Q_LEN     = 64
    MAX_A_LEN     = 10
    IMG_SIZE      = 260
    IMG_FEAT_DIM  = 1408
    IMG_SPATIAL_C = 1408
    TXT_FEAT_DIM  = 768
    FUSION_DIM    = 512
    LSTM_HIDDEN   = 512
    LSTM_LAYERS   = 2
    TF_HEADS      = 8
    TF_LAYERS     = 4
    DROPOUT       = 0.3

cfg_a = CfgA()
DEVICE_A = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_A = {
    "A1": "checkpoints/A1/A1_LSTM_best.pth",
    "A2": "checkpoints/A2/A2_Transformer_best.pth",
}
VOCAB_PATH_A = "checkpoints/A1/vocab.json"

# ── Model classes ─────────────
class ImageEncoderA(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b2', pretrained=False, num_classes=0, global_pool=''
        )
        self.proj_global = nn.Sequential(
            nn.Linear(cfg_a.IMG_FEAT_DIM, cfg_a.FUSION_DIM),
            nn.LayerNorm(cfg_a.FUSION_DIM), nn.ReLU(), nn.Dropout(cfg_a.DROPOUT),
        )
        self.proj_spatial = nn.Sequential(
            nn.Linear(cfg_a.IMG_SPATIAL_C, cfg_a.FUSION_DIM),
            nn.LayerNorm(cfg_a.FUSION_DIM), nn.ReLU(), nn.Dropout(cfg_a.DROPOUT),
        )
        self.spatial_pos_emb = nn.Parameter(torch.randn(1, 100, cfg_a.FUSION_DIM) * 0.02)

    def forward(self, images):
        feat = self.backbone(images)
        return self.proj_global(feat.mean(dim=[2, 3]))

    def forward_spatial(self, images):
        feat = self.backbone(images)
        B, C, H, W = feat.shape
        n = H * W
        feat = feat.permute(0, 2, 3, 1).reshape(B, n, C)
        projected = self.proj_spatial(feat)
        return projected + self.spatial_pos_emb[:, :n, :]


class TextEncoderA(nn.Module):
    def __init__(self):
        super().__init__()
        self.phobert = AutoModel.from_pretrained('vinai/phobert-base')
        self.proj = nn.Sequential(
            nn.Linear(cfg_a.TXT_FEAT_DIM, cfg_a.FUSION_DIM),
            nn.LayerNorm(cfg_a.FUSION_DIM), nn.ReLU(), nn.Dropout(cfg_a.DROPOUT),
        )

    def forward(self, input_ids, attention_mask):
        out = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        summed = (hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return self.proj(summed / counts)


class FusionModuleA(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(cfg_a.FUSION_DIM * 2, cfg_a.FUSION_DIM),
            nn.LayerNorm(cfg_a.FUSION_DIM), nn.ReLU(), nn.Dropout(cfg_a.DROPOUT),
            nn.Linear(cfg_a.FUSION_DIM, cfg_a.FUSION_DIM),
        )

    def forward(self, img_feat, txt_feat):
        return self.fusion(torch.cat([img_feat, txt_feat], dim=-1))


class LSTMDecoderA(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.hidden_dim = cfg_a.LSTM_HIDDEN
        self.num_layers = cfg_a.LSTM_LAYERS
        self.embedding   = nn.Embedding(vocab_size, 256, padding_idx=0)
        self.init_h      = nn.Linear(cfg_a.FUSION_DIM, cfg_a.LSTM_HIDDEN * cfg_a.LSTM_LAYERS)
        self.init_c      = nn.Linear(cfg_a.FUSION_DIM, cfg_a.LSTM_HIDDEN * cfg_a.LSTM_LAYERS)
        self.lstm        = nn.LSTM(256, cfg_a.LSTM_HIDDEN, cfg_a.LSTM_LAYERS,
                                   batch_first=True,
                                   dropout=cfg_a.DROPOUT if cfg_a.LSTM_LAYERS > 1 else 0)
        self.output_proj = nn.Linear(cfg_a.LSTM_HIDDEN, vocab_size)
        self.dropout     = nn.Dropout(cfg_a.DROPOUT)

    def forward(self, fusion_feat, answer_ids=None, teacher_forcing=False):
        batch = fusion_feat.size(0)
        h0 = self.init_h(fusion_feat).view(batch, self.num_layers, self.hidden_dim).permute(1,0,2).contiguous()
        c0 = self.init_c(fusion_feat).view(batch, self.num_layers, self.hidden_dim).permute(1,0,2).contiguous()
        current = torch.full((batch, 1), 1, dtype=torch.long, device=fusion_feat.device)
        h, c = h0, c0
        logits_all = []
        finished = torch.zeros(batch, dtype=torch.bool, device=fusion_feat.device)
        for _ in range(cfg_a.MAX_A_LEN + 1):
            embed = self.dropout(self.embedding(current))
            out, (h, c) = self.lstm(embed, (h, c))
            logit = self.output_proj(out)
            logits_all.append(logit)
            next_token = logit.argmax(dim=-1).squeeze(-1)
            finished = finished | (next_token == 2)
            if finished.all():
                break
            current = next_token.unsqueeze(-1)
        return torch.cat(logits_all, dim=1)


class PositionalEncodingA(nn.Module):
    def __init__(self, d_model, max_len=64, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerDecoderA(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed_dim = cfg_a.FUSION_DIM
        self.embedding = nn.Embedding(vocab_size, cfg_a.FUSION_DIM, padding_idx=0)
        self.pos_enc   = PositionalEncodingA(cfg_a.FUSION_DIM, max_len=cfg_a.MAX_A_LEN + 10)
        decoder_layer  = nn.TransformerDecoderLayer(
            d_model=cfg_a.FUSION_DIM, nhead=cfg_a.TF_HEADS,
            dim_feedforward=cfg_a.FUSION_DIM * 4,
            dropout=cfg_a.DROPOUT, batch_first=True,
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=cfg_a.TF_LAYERS)
        self.output_proj = nn.Linear(cfg_a.FUSION_DIM, vocab_size)

    @staticmethod
    def _causal_mask(size, device):
        return torch.triu(torch.ones(size, size, device=device), diagonal=1).bool()

    def forward(self, memory, answer_ids=None, teacher_forcing=False):
        batch   = memory.size(0)
        device  = memory.device
        current  = torch.full((batch, 1), 1, dtype=torch.long, device=device)
        outputs  = []
        finished = torch.zeros(batch, dtype=torch.bool, device=device)
        for _ in range(cfg_a.MAX_A_LEN + 1):
            seq_len  = current.size(1)
            pad_mask = (current == 0)
            tgt = self.pos_enc(self.embedding(current) * math.sqrt(self.embed_dim))
            out = self.decoder(
                tgt=tgt, memory=memory,
                tgt_mask=self._causal_mask(seq_len, device),
                tgt_key_padding_mask=pad_mask,
            )
            logit      = self.output_proj(out[:, -1:, :])
            outputs.append(logit)
            next_token = logit.argmax(dim=-1).squeeze(-1)
            finished   = finished | (next_token == 2)
            if finished.all():
                break
            current = torch.cat([current, next_token.unsqueeze(-1)], dim=1)
        return torch.cat(outputs, dim=1)


class VQAModelA(nn.Module):
    def __init__(self, vocab_size, decoder_type='lstm'):
        super().__init__()
        self.decoder_type = decoder_type
        self.img_encoder  = ImageEncoderA()
        self.txt_encoder  = TextEncoderA()
        self.fusion       = FusionModuleA()
        if decoder_type == 'lstm':
            self.decoder = LSTMDecoderA(vocab_size=vocab_size)
        else:
            self.decoder = TransformerDecoderA(vocab_size=vocab_size)

    def forward(self, images, input_ids, attention_mask,
                answer_ids=None, teacher_forcing=False):
        txt_feat = self.txt_encoder(input_ids, attention_mask)
        if self.decoder_type == 'lstm':
            img_feat    = self.img_encoder(images)
            fusion_feat = self.fusion(img_feat, txt_feat)
            return self.decoder(fusion_feat)
        else:
            spatial = self.img_encoder.forward_spatial(images)
            memory  = torch.cat([txt_feat.unsqueeze(1), spatial], dim=1)
            return self.decoder(memory)


# ── Vocab ──────────
with open(VOCAB_PATH_A, encoding='utf-8') as f:
    _vocab_data = json.load(f)

_word2idx_a = _vocab_data['word2idx']
_idx2word_a = {int(k): v for k, v in _vocab_data['idx2word'].items()}
_n_words_a  = _vocab_data['n_words']

def decode_answer(indices):
    words = []
    for idx in indices:
        word = _idx2word_a.get(idx, '<UNK>')
        if word == '<EOS>': break
        if word in ('<SOS>', '<PAD>', '<UNK>'): continue
        words.append(word)
    return ' '.join(words)

# ── Tokenizer ────────
print("Loading PhoBERT tokenizer...")
_tokenizer_a = AutoTokenizer.from_pretrained('vinai/phobert-base')
print("✓ PhoBERT tokenizer loaded!")

# ── Transform ────────
_transform_a = transforms.Compose([
    transforms.Resize((cfg_a.IMG_SIZE, cfg_a.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Load A1 ─────────
print("\nLoading A1 (LSTM decoder)...")
model_a1 = VQAModelA(vocab_size=_n_words_a, decoder_type='lstm').to(DEVICE_A)
ckpt_a1  = torch.load(CKPT_A["A1"], map_location=DEVICE_A)
model_a1.load_state_dict(ckpt_a1['model_state'])
model_a1.eval()
print(f"✓ A1 loaded! epoch={ckpt_a1.get('epoch','?')} val_acc={ckpt_a1.get('val_acc','?')}")

# ── Load A2 ───────
print("\nLoading A2 (Transformer decoder)...")
model_a2 = VQAModelA(vocab_size=_n_words_a, decoder_type='transformer').to(DEVICE_A)
ckpt_a2  = torch.load(CKPT_A["A2"], map_location=DEVICE_A)
model_a2.load_state_dict(ckpt_a2['model_state'])
model_a2.eval()
print(f"✓ A2 loaded! epoch={ckpt_a2.get('epoch','?')} val_acc={ckpt_a2.get('val_acc','?')}")

print("\n✓ A1 + A2 sẵn sàng!")

Loading PhoBERT tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

✓ A1 loaded! epoch=26 val_acc=51.985559566786996

Loading A2 (Transformer decoder)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ A2 loaded! epoch=28 val_acc=50.90252707581227

✓ A1 + A2 sẵn sàng!


In [ ]:
def predict_a(model, image: Image.Image, question: str, model_name: str):
    img_tensor = _transform_a(image.convert('RGB')).unsqueeze(0).to(DEVICE_A)

    enc = _tokenizer_a(
        question,
        max_length=cfg_a.MAX_Q_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )
    input_ids      = enc['input_ids'].to(DEVICE_A)
    attention_mask = enc['attention_mask'].to(DEVICE_A)

    with torch.no_grad():
        logits   = model(img_tensor, input_ids, attention_mask)
        pred_ids = logits.argmax(dim=-1)[0]

    answer = decode_answer(pred_ids.tolist())
    print(f"[{model_name}] answer: '{answer}'")

    if not answer:
        answer = "(không có câu trả lời)"

    probs       = torch.softmax(logits[0, 0], dim=-1)
    top3_v, top3_i = probs.topk(3)
    conf        = float(top3_v[0])
    top3_words  = [_idx2word_a.get(int(top3_i[k]), '?') for k in range(1, 3)]

    return [(answer, conf), (top3_words[0], float(top3_v[1])), (top3_words[1], float(top3_v[2]))]


# Thêm A1/A2 vào _cache để Gradio dùng chung
_cache["A1"] = (model_a1, None)
_cache["A2"] = (model_a2, None)
print("✓ _cache đã có: A1, A2, B1, B2")

✓ _cache đã có: A1, A2, B1, B2


In [ ]:
import gradio as gr
import time

def classify_question(question: str) -> str:
    q = question.lower()
    if any(w in q for w in ["có", "không", "phải", "đúng", "sai"]):
        return "Yes/No"
    if any(w in q for w in ["mấy", "bao nhiêu", "số lượng"]):
        return "Đếm số"
    if any(w in q for w in ["màu"]):
        return "Thuộc tính"
    if any(w in q for w in ["gì", "là gì", "tên"]):
        return "Nhận dạng"
    if any(w in q for w in ["ở đâu", "vị trí", "bên", "trên", "dưới"]):
        return "Không gian"
    return "Mở"

def run_inference(image, question, model_choice):
    if image is None:
        return "⚠️ Chưa upload ảnh", "", "", gr.update(visible=False)
    if not question.strip():
        return "⚠️ Chưa nhập câu hỏi", "", "", gr.update(visible=False)

    t0 = time.time()

    if model_choice in ("A1", "A2"):
        model, _ = _cache[model_choice]
        top3 = predict_a(model, image, question, model_choice)
        direction = "Hướng A (tự train)"
        decoder   = "LSTM decoder" if model_choice == "A1" else "Transformer decoder"
    else:
        model, aux = _cache[model_choice]
        top3 = predict(model, aux, image, question, model_choice)
        direction = "Hướng B (pretrained)"
        decoder   = "Zero-shot" if model_choice == "B1" else "Fine-tuned"

    elapsed = time.time() - t0
    answer  = top3[0][0]
    conf    = f"{top3[0][1]*100:.0f}%"
    qtype   = classify_question(question)

    info = (
        f"**Mô hình:** {model_choice} ({decoder})  |  "
        f"**Hướng:** {direction}  |  "
        f"**Confidence:** {conf}  |  "
        f"**Thời gian:** {elapsed:.1f}s  |  "
        f"**Loại câu hỏi:** {qtype}"
    )
    return answer, info, "", gr.update(visible=True)


def run_all_inference(image, question):
    if image is None:
        return (
            "⚠️ Chưa upload ảnh", "", "", "",
            "", "", "", "",
            "", "", "", "",
            "", "", "", "",
            gr.update(visible=False),
        )
    if not question.strip():
        return (
            "⚠️ Chưa nhập câu hỏi", "", "", "",
            "", "", "", "",
            "", "", "", "",
            "", "", "", "",
            gr.update(visible=False),
        )

    results = {}
    for m in ["A1", "A2", "B1", "B2"]:
        t0 = time.time()
        if m in ("A1", "A2"):
            model, _ = _cache[m]
            top3 = predict_a(model, image, question, m)
            direction = "Hướng A (tự train)"
            decoder   = "LSTM" if m == "A1" else "Transformer"
        else:
            model, aux = _cache[m]
            top3 = predict(model, aux, image, question, m)
            direction = "Hướng B (pretrained)"
            decoder   = "Zero-shot" if m == "B1" else "Fine-tuned"
        elapsed = time.time() - t0
        conf = f"{top3[0][1]*100:.0f}%"
        qtype = classify_question(question)
        results[m] = {
            "answer": top3[0][0] if top3[0][0] else "(không có)",
            "info": (
                f"**{m}** ({decoder}) | {direction} | "
                f"Conf: {conf} | {elapsed:.1f}s | {qtype}"
            ),
        }

    return (
        results["A1"]["answer"], results["A1"]["info"],
        results["A2"]["answer"], results["A2"]["info"],
        results["B1"]["answer"], results["B1"]["info"],
        results["B2"]["answer"], results["B2"]["info"],
        gr.update(visible=True),
    )


PURPLE_CSS = """
:root {
    --primary-hue: 270;
}
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
.gr-button-primary {
    background: linear-gradient(135deg, #7c3aed, #9f5cf7) !important;
    border: none !important;
}
#title-md h1 { color: #7c3aed; }
"""

with gr.Blocks(title="VQA Tiếng Việt", theme=gr.themes.Default(
    primary_hue=gr.themes.colors.purple,
    secondary_hue=gr.themes.colors.purple,
    neutral_hue=gr.themes.colors.slate,
)) as demo:

    gr.Markdown("# 🍜 VQA Tiếng Việt — Món ăn Việt Nam\nHỏi về ảnh món ăn bằng tiếng Việt", elem_id="title-md")

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Ảnh đầu vào", height=300)

        with gr.Column(scale=1):
            question_input = gr.Textbox(
                label="Câu hỏi tiếng Việt",
                placeholder="Đây là món ăn gì?",
                lines=2,
            )
            model_choice = gr.Radio(
                choices=["A1", "A2", "B1", "B2"],
                value="B2",
                label="Chọn mô hình",
                info="A1: CNN+LSTM | A2: CNN+Transformer | B1: PaliGemma zero-shot | B2: PaliGemma fine-tuned",
            )
            with gr.Row():
                run_btn     = gr.Button("Hỏi mô hình", variant="primary", size="lg")
                run_all_btn = gr.Button("🔀 So sánh tất cả", variant="secondary", size="lg")

    # ── Kết quả 1 mô hình ──────────
    single_group = gr.Group(visible=False)
    with single_group:
        gr.Markdown("### Kết quả")
        answer_out = gr.Textbox(label="Câu trả lời", interactive=False, text_align="center")
        info_out   = gr.Markdown()
        _hidden    = gr.Textbox(visible=False)

    # ── Kết quả so sánh tất cả ────────
    all_group = gr.Group(visible=False)
    with all_group:
        gr.Markdown("### 🔀 So sánh tất cả mô hình")
        with gr.Row():
            with gr.Column():
                gr.Markdown("#### A1 — LSTM")
                a1_ans  = gr.Textbox(label="Câu trả lời", interactive=False, text_align="center")
                a1_info = gr.Markdown()
            with gr.Column():
                gr.Markdown("#### A2 — Transformer")
                a2_ans  = gr.Textbox(label="Câu trả lời", interactive=False, text_align="center")
                a2_info = gr.Markdown()
        with gr.Row():
            with gr.Column():
                gr.Markdown("#### B1 — PaliGemma Zero-shot")
                b1_ans  = gr.Textbox(label="Câu trả lời", interactive=False, text_align="center")
                b1_info = gr.Markdown()
            with gr.Column():
                gr.Markdown("#### B2 — PaliGemma Fine-tuned")
                b2_ans  = gr.Textbox(label="Câu trả lời", interactive=False, text_align="center")
                b2_info = gr.Markdown()

    # ── Events ─────────
    run_btn.click(
        fn=run_inference,
        inputs=[image_input, question_input, model_choice],
        outputs=[answer_out, info_out, _hidden, single_group],
    )

    run_all_btn.click(
        fn=run_all_inference,
        inputs=[image_input, question_input],
        outputs=[
            a1_ans, a1_info,
            a2_ans, a2_info,
            b1_ans, b1_info,
            b2_ans, b2_info,
            all_group,
        ],
    )

demo.launch(share=True)

/tmp/ipykernel_7101/2811512358.py:117: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="VQA Tiếng Việt", theme=gr.themes.Default(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3824d87b24dc4f0092.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
